In [3]:
# ============================================================
# Medicaid Expenditure Projection Model
# National + New York State | FY 2013–2025 → Forecast 2026–2035
# Author: Asmita Thapa
# Date: April 2026
# ============================================================

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openpyxl
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.optimize import minimize
from scipy import stats

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("openpyxl:", openpyxl.__version__)
print("sklearn: OK")
print("scipy: OK")
print("✅ All libraries loaded successfully")

numpy: 2.3.5
pandas: 2.3.3
openpyxl: 3.1.5
sklearn: OK
scipy: OK
✅ All libraries loaded successfully


In [5]:
import os
import warnings
warnings.filterwarnings('ignore')

# Set matplotlib backend
import matplotlib
matplotlib.use('Agg')

# Plot styling
import matplotlib.pyplot as plt
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 150,
})

# ── Folder paths ──────────────────────────────────────────────
DATA_DIR = "C:/Users/User/Documents/EMRTS_ProjectV2"  
OUT_DIR  = os.path.join(DATA_DIR, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

print("Data folder:", DATA_DIR)
print("Output folder:", OUT_DIR)
print("Output folder created:", os.path.exists(OUT_DIR))

Data folder: C:/Users/User/Documents/EMRTS_ProjectV2
Output folder: C:/Users/User/Documents/EMRTS_ProjectV2\outputs
Output folder created: True


In [6]:
# ============================================================
# DATA INGESTION & CLEANING
# ============================================================

# These are words that appear in non-data rows (footnotes, totals)
# We use this list to skip those rows during extraction
SKIP_PREFIXES = [
    'total', 'grand total', '¹', '²', 'information', 'all info',
    'recoveries are', 'mfcu grant', 'investigations are',
    'staff on', 'footnote', 'fy 20',
    '1 ', '2 ', '3 ', '4 ', '5 ', '6 ',
]

records = []

for yr in range(2013, 2026):
    path = os.path.join(DATA_DIR, f"FY_{yr}_MFCU_Statistical_Chart.xlsx")
    
    # Check file exists
    if not os.path.exists(path):
        print(f"WARNING: File not found — FY {yr}")
        continue

    wb = openpyxl.load_workbook(path, read_only=True)
    
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        rows = list(ws.iter_rows(values_only=True))

        # Find the header row dynamically
        # Different years use 'State' or 'State1' as the column name
        header_idx = None
        for i, row in enumerate(rows):
            if row and any(
                str(c).strip().lower() in ('state', 'states', 'state1')
                for c in row if c
            ):
                header_idx = i
                break
        if header_idx is None:
            continue

        headers = [str(c).strip() if c else '' for c in rows[header_idx]]

        # Find the 'Total Medicaid Expenditures' column
        med_col = next(
            (j for j, h in enumerate(headers) if 'total medicaid' in h.lower()),
            None
        )
        if med_col is None:
            continue

        # Extract state-level rows only
        for row in rows[header_idx + 1:]:
            if not row or not row[0]:
                continue
            state_raw = str(row[0]).strip()

            # Skip footnotes, totals, long annotation rows
            if len(state_raw) > 60:
                continue
            state_low = state_raw.lower()
            if any(state_low.startswith(p) for p in SKIP_PREFIXES):
                continue

            val = row[med_col]
            if val is None:
                continue
            try:
                val_f = float(str(val).replace(',', '').replace('$', ''))
                if val_f > 0:
                    records.append({
                        'FY': yr,
                        'State': state_raw.title(),
                        'Medicaid_Expenditure': val_f
                    })
            except (ValueError, TypeError):
                continue

df_raw = pd.DataFrame(records)

print(f"Total records loaded: {len(df_raw)}")
print(f"Years found: {sorted(df_raw['FY'].unique())}")
print(f"\nStates per fiscal year:")
print(df_raw.groupby('FY')['State'].count().to_string())
print(f"\nMissing values: {df_raw['Medicaid_Expenditure'].isna().sum()}")
print(f"Duplicate state-year pairs: {df_raw.duplicated(['FY','State']).sum()}")
print("\n✅ Data cleaning complete")

Total records loaded: 670
Years found: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

States per fiscal year:
FY
2013    50
2014    50
2015    50
2016    50
2017    50
2018    50
2019    52
2020    53
2021    53
2022    53
2023    53
2024    53
2025    53

Missing values: 0
Duplicate state-year pairs: 0

✅ Data cleaning complete


In [7]:
# ============================================================
# BUILD TIME SERIES (National + New York)
# ============================================================

# National: sum all jurisdictions per year
national_ts = (
    df_raw.groupby('FY')['Medicaid_Expenditure']
    .sum()
    .reset_index()
    .rename(columns={'FY': 'Year', 'Medicaid_Expenditure': 'Expenditure'})
    .sort_values('Year')
    .reset_index(drop=True)
)

# New York: filter NY rows then sum per year
ny_ts = (
    df_raw[df_raw['State'].str.contains('New York', case=False)]
    .groupby('FY')['Medicaid_Expenditure']
    .sum()
    .reset_index()
    .rename(columns={'FY': 'Year', 'Medicaid_Expenditure': 'Expenditure'})
    .sort_values('Year')
    .reset_index(drop=True)
)

# Year-over-year growth
national_ts['YoY_Growth_pct'] = national_ts['Expenditure'].pct_change() * 100
ny_ts['YoY_Growth_pct'] = ny_ts['Expenditure'].pct_change() * 100

# Print results
print("National Medicaid Expenditures (Billions USD):")
print("-" * 40)
for _, r in national_ts.iterrows():
    print(f"  FY {int(r.Year)}: ${r.Expenditure/1e9:.2f}B")

print("\nNew York Medicaid Expenditures (Billions USD):")
print("-" * 40)
for _, r in ny_ts.iterrows():
    print(f"  FY {int(r.Year)}: ${r.Expenditure/1e9:.2f}B")

print(f"\nNational average YoY growth: {national_ts['YoY_Growth_pct'].mean():.1f}%")
print(f"New York average YoY growth: {ny_ts['YoY_Growth_pct'].mean():.1f}%")

print("\n✅ Time series built successfully")

National Medicaid Expenditures (Billions USD):
----------------------------------------
  FY 2013: $453.08B
  FY 2014: $488.24B
  FY 2015: $548.19B
  FY 2016: $571.23B
  FY 2017: $596.43B
  FY 2018: $611.98B
  FY 2019: $614.91B
  FY 2020: $678.89B
  FY 2021: $740.29B
  FY 2022: $823.87B
  FY 2023: $893.14B
  FY 2024: $948.85B
  FY 2025: $1031.70B

New York Medicaid Expenditures (Billions USD):
----------------------------------------
  FY 2013: $54.19B
  FY 2014: $53.92B
  FY 2015: $59.68B
  FY 2016: $62.91B
  FY 2017: $78.56B
  FY 2018: $75.26B
  FY 2019: $60.21B
  FY 2020: $72.82B
  FY 2021: $73.27B
  FY 2022: $82.56B
  FY 2023: $94.60B
  FY 2024: $98.18B
  FY 2025: $102.70B

National average YoY growth: 7.2%
New York average YoY growth: 6.1%

✅ Time series built successfully


In [8]:
# ============================================================
# EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Medicaid Expenditure — Exploratory Data Analysis (FY 2013–2025)', 
             fontsize=14, fontweight='bold', y=0.98)

# ── Panel 1: National historical trend ───────────────────────
ax1 = axes[0, 0]
ax1.plot(national_ts['Year'], national_ts['Expenditure']/1e9,
         marker='o', color='#1f77b4', linewidth=2, markersize=7)
ax1.fill_between(national_ts['Year'], national_ts['Expenditure']/1e9,
                 alpha=0.1, color='#1f77b4')
ax1.set_title('National Total Medicaid Expenditures')
ax1.set_xlabel('Fiscal Year')
ax1.set_ylabel('Expenditure (Billions USD)')
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

# ── Panel 2: New York historical trend ───────────────────────
ax2 = axes[0, 1]
ax2.plot(ny_ts['Year'], ny_ts['Expenditure']/1e9,
         marker='s', color='#d62728', linewidth=2, markersize=7)
ax2.fill_between(ny_ts['Year'], ny_ts['Expenditure']/1e9,
                 alpha=0.1, color='#d62728')
ax2.set_title('New York State Medicaid Expenditures')
ax2.set_xlabel('Fiscal Year')
ax2.set_ylabel('Expenditure (Billions USD)')
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

# ── Panel 3: Year-over-year growth rates ─────────────────────
ax3 = axes[1, 0]
nat_growth = national_ts.dropna(subset=['YoY_Growth_pct'])
ny_growth  = ny_ts.dropna(subset=['YoY_Growth_pct'])
x = np.arange(len(nat_growth))
width = 0.35
ax3.bar(x - width/2, nat_growth['YoY_Growth_pct'], width,
        label='National', color='#1f77b4', alpha=0.8)
ax3.bar(x + width/2, ny_growth['YoY_Growth_pct'], width,
        label='New York', color='#d62728', alpha=0.8)
ax3.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax3.set_title('Year-over-Year Growth Rate (%)')
ax3.set_xlabel('Fiscal Year')
ax3.set_ylabel('Growth Rate (%)')
ax3.set_xticks(x)
ax3.set_xticklabels(nat_growth['Year'].astype(int), rotation=45)
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# ── Panel 4: NY share of national ────────────────────────────
ax4 = axes[1, 1]
ny_share = ny_ts['Expenditure'].values / national_ts['Expenditure'].values * 100
ax4.plot(national_ts['Year'], ny_share,
         marker='D', color='#2ca02c', linewidth=2, markersize=7)
ax4.set_title("New York's Share of National Medicaid Expenditures")
ax4.set_xlabel('Fiscal Year')
ax4.set_ylabel('NY Share (%)')
ax4.grid(True, alpha=0.3)
ax4.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax4.set_ylim(0, 20)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_EDA_Overview.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA chart saved to outputs folder")

✅ EDA chart saved to outputs folder


In [9]:
# ============================================================
# DESCRIPTIVE STATISTICS + TIME SERIES DECOMPOSITION
# ============================================================

# ── Descriptive Statistics ───────────────────────────────────
print("=" * 55)
print("DESCRIPTIVE STATISTICS")
print("=" * 55)

for label, ts in [("National", national_ts), ("New York", ny_ts)]:
    exp = ts['Expenditure'] / 1e9
    growth = ts['YoY_Growth_pct'].dropna()
    print(f"\n  {label} (Billions USD):")
    print(f"    Min:               ${exp.min():.2f}B  (FY {int(ts.loc[exp.idxmin(),'Year'])})")
    print(f"    Max:               ${exp.max():.2f}B  (FY {int(ts.loc[exp.idxmax(),'Year'])})")
    print(f"    Mean:              ${exp.mean():.2f}B")
    print(f"    Median:            ${exp.median():.2f}B")
    print(f"    Std Deviation:     ${exp.std():.2f}B")
    print(f"    Avg YoY Growth:    {growth.mean():.1f}%")
    print(f"    Max YoY Growth:    {growth.max():.1f}%  (FY {int(ts.loc[growth.idxmax(),'Year'])})")
    print(f"    Min YoY Growth:    {growth.min():.1f}%  (FY {int(ts.loc[growth.idxmin(),'Year'])})")

# ── Time Series Decomposition ────────────────────────────────
# We decompose each series into 3 components:
# Trend    = smoothed long-term direction (using rolling mean)
# Seasonal = repeating pattern (not applicable for annual data
#            so we show it as flat/zero)
# Residual = what's left after removing trend (noise/anomalies)

def decompose_ts(ts_df):
    y = ts_df['Expenditure'].values / 1e9
    years = ts_df['Year'].values

    # Trend: centered rolling mean with window=3
    trend = pd.Series(y).rolling(window=3, center=True).mean().values

    # Residual: actual minus trend
    residual = np.where(~np.isnan(trend), y - trend, np.nan)

    return years, y, trend, residual

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Time Series Decomposition — National & New York\n(FY 2013–2025)',
             fontsize=14, fontweight='bold')

for row, (label, ts, color) in enumerate([
    ("National", national_ts, '#1f77b4'),
    ("New York", ny_ts,       '#d62728')
]):
    years, y, trend, residual = decompose_ts(ts)

    # Original + Trend
    ax = axes[row, 0]
    ax.plot(years, y,     marker='o', color=color,   linewidth=2,
            markersize=5, label='Actual')
    ax.plot(years, trend, marker='',  color='black',  linewidth=2,
            linestyle='--', label='Trend (3yr avg)')
    ax.set_title(f'{label} — Actual vs Trend')
    ax.set_xlabel('Fiscal Year')
    ax.set_ylabel('Billions USD')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    # Trend only
    ax = axes[row, 1]
    ax.plot(years, trend, marker='o', color='black', linewidth=2, markersize=5)
    ax.set_title(f'{label} — Trend Component')
    ax.set_xlabel('Fiscal Year')
    ax.set_ylabel('Billions USD')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    # Residual
    ax = axes[row, 2]
    ax.bar(years, residual,
           color=[color if r >= 0 else '#ff7f0e' for r in
                  [x if x == x else 0 for x in residual]],
           alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'{label} — Residual Component')
    ax.set_xlabel('Fiscal Year')
    ax.set_ylabel('Deviation (Billions USD)')
    ax.grid(True, alpha=0.3, axis='y')
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/02_Decomposition.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Decomposition chart saved")

DESCRIPTIVE STATISTICS

  National (Billions USD):
    Min:               $453.08B  (FY 2013)
    Max:               $1031.70B  (FY 2025)
    Mean:              $692.37B
    Median:            $614.91B
    Std Deviation:     $181.90B
    Avg YoY Growth:    7.2%
    Max YoY Growth:    12.3%  (FY 2015)
    Min YoY Growth:    0.5%  (FY 2019)

  New York (Billions USD):
    Min:               $53.92B  (FY 2014)
    Max:               $102.70B  (FY 2025)
    Mean:              $74.53B
    Median:            $73.27B
    Std Deviation:     $16.43B
    Avg YoY Growth:    6.1%
    Max YoY Growth:    24.9%  (FY 2017)
    Min YoY Growth:    -20.0%  (FY 2019)

✅ Decomposition chart saved


In [10]:
# ============================================================
# FORECASTING MODELS
# ============================================================

# ── Setup: train/test split and helper functions ─────────────

FUTURE_YEARS = np.arange(2026, 2036)  # 2026 to 2035
TRAIN_END    = 2021  # train on 2013–2021, test on 2022–2025

def train_test_split_ts(ts_df, train_end):
    train = ts_df[ts_df['Year'] <= train_end].copy()
    test  = ts_df[ts_df['Year'] >  train_end].copy()
    return train, test

def compute_metrics(actual, predicted, label=""):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    print(f"    {label}:")
    print(f"      MAE:  ${mae/1e9:.2f}B")
    print(f"      RMSE: ${rmse/1e9:.2f}B")
    print(f"      MAPE: {mape:.1f}%")
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

print("✅ Setup complete")
print(f"   Training period: FY 2013–{TRAIN_END}")
print(f"   Test period:     FY {TRAIN_END+1}–2025")
print(f"   Forecast period: FY 2026–2035")

✅ Setup complete
   Training period: FY 2013–2021
   Test period:     FY 2022–2025
   Forecast period: FY 2026–2035


In [13]:
# ============================================================
# MODEL 1: LINEAR REGRESSION
# ============================================================

def fit_linear(ts_df, train_end, future_years):
    train, test = train_test_split_ts(ts_df, train_end)
    
    X_train = train['Year'].values.reshape(-1, 1)
    y_train = train['Expenditure'].values
    X_test  = test['Year'].values.reshape(-1, 1)
    y_test  = test['Expenditure'].values

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_hat_train = model.predict(X_train)
    residuals   = y_train - y_hat_train
    se          = np.std(residuals, ddof=2)
    n           = len(train)

    test_pred = model.predict(X_test)
    metrics   = compute_metrics(y_test, test_pred, "Linear Regression")

    X_future = future_years.reshape(-1, 1)
    y_future = model.predict(X_future)
    x_mean   = X_train.mean()
    ss_xx    = ((X_train - x_mean) ** 2).sum()

    ci_width = []
    for xf in future_years:
        leverage = 1 + 1/n + (xf - x_mean)**2 / ss_xx
        ci_width.append(stats.t.ppf(0.975, df=n-2) * se * np.sqrt(leverage))
    ci_width = np.array(ci_width)

    forecast_df = pd.DataFrame({
        'Year':     future_years,
        'Forecast': y_future,
        'Lower_95': y_future - ci_width,
        'Upper_95': y_future + ci_width,
    })
    return model, metrics, forecast_df, test_pred

print("=" * 45)
print("MODEL 1: LINEAR REGRESSION")
print("=" * 45)

print("\n  National:")
nat_lin_model, nat_lin_metrics, nat_lin_fc, nat_lin_test_pred = fit_linear(national_ts, TRAIN_END, FUTURE_YEARS)

print("\n  New York:")
ny_lin_model, ny_lin_metrics, ny_lin_fc, ny_lin_test_pred = fit_linear(ny_ts, TRAIN_END, FUTURE_YEARS)

print("\n  National forecast 2026-2035:")
for _, r in nat_lin_fc.iterrows():
    print(f"    {int(r.Year)}: ${r.Forecast/1e9:.2f}B  [${r.Lower_95/1e9:.2f}B - ${r.Upper_95/1e9:.2f}B]")

print("\n  New York forecast 2026-2035:")
for _, r in ny_lin_fc.iterrows():
    print(f"    {int(r.Year)}: ${r.Forecast/1e9:.2f}B  [${r.Lower_95/1e9:.2f}B - ${r.Upper_95/1e9:.2f}B]")

print("\n✅ Model 1 complete")

MODEL 1: LINEAR REGRESSION

  National:
    Linear Regression:
      MAE:  $129.85B
      RMSE: $136.18B
      MAPE: 13.8%

  New York:
    Linear Regression:
      MAE:  $13.00B
      RMSE: $13.90B
      MAPE: 13.4%

  National forecast 2026-2035:
    2026: $873.49B  [$796.93B - $950.05B]
    2027: $905.07B  [$823.74B - $986.41B]
    2028: $936.66B  [$850.35B - $1022.97B]
    2029: $968.24B  [$876.79B - $1059.69B]
    2030: $999.82B  [$903.10B - $1096.54B]
    2031: $1031.40B  [$929.29B - $1133.52B]
    2032: $1062.99B  [$955.39B - $1170.59B]
    2033: $1094.57B  [$981.39B - $1207.74B]
    2034: $1126.15B  [$1007.33B - $1244.97B]
    2035: $1157.73B  [$1033.20B - $1282.27B]

  New York forecast 2026-2035:
    2026: $87.61B  [$61.27B - $113.95B]
    2027: $90.05B  [$62.06B - $118.03B]
    2028: $92.49B  [$62.79B - $122.18B]
    2029: $94.93B  [$63.47B - $126.39B]
    2030: $97.37B  [$64.09B - $130.65B]
    2031: $99.81B  [$64.68B - $134.94B]
    2032: $102.25B  [$65.23B - $139.27B]
   

In [14]:
# ============================================================
# MODEL 2: POLYNOMIAL REGRESSION (DEGREE 2)
# ============================================================

def fit_polynomial(ts_df, train_end, future_years, degree=2):
    train, test = train_test_split_ts(ts_df, train_end)

    X_train = train['Year'].values.reshape(-1, 1)
    y_train = train['Expenditure'].values
    X_test  = test['Year'].values.reshape(-1, 1)
    y_test  = test['Expenditure'].values

    # Pipeline: first create polynomial features, then fit linear regression
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train, y_train)

    # Residuals for confidence interval
    y_hat_train = model.predict(X_train)
    residuals   = y_train - y_hat_train
    se          = np.std(residuals, ddof=degree+1)
    n           = len(train)
    t_crit      = stats.t.ppf(0.975, df=n-degree-1)

    # Test predictions and metrics
    test_pred = model.predict(X_test)
    metrics   = compute_metrics(y_test, test_pred, "Polynomial Regression")

    # Future forecast
    X_future = future_years.reshape(-1, 1)
    y_future = model.predict(X_future)
    ci_width = t_crit * se * np.sqrt(1 + 1/n)

    forecast_df = pd.DataFrame({
        'Year':     future_years,
        'Forecast': y_future,
        'Lower_95': y_future - ci_width,
        'Upper_95': y_future + ci_width,
    })
    return model, metrics, forecast_df, test_pred

print("=" * 45)
print("MODEL 2: POLYNOMIAL REGRESSION (DEGREE 2)")
print("=" * 45)

print("\n  National:")
nat_poly_model, nat_poly_metrics, nat_poly_fc, nat_poly_test_pred = fit_polynomial(national_ts, TRAIN_END, FUTURE_YEARS)

print("\n  New York:")
ny_poly_model, ny_poly_metrics, ny_poly_fc, ny_poly_test_pred = fit_polynomial(ny_ts, TRAIN_END, FUTURE_YEARS)

print("\n  National forecast 2026-2035:")
for _, r in nat_poly_fc.iterrows():
    print(f"    {int(r.Year)}: ${r.Forecast/1e9:.2f}B  [${r.Lower_95/1e9:.2f}B - ${r.Upper_95/1e9:.2f}B]")

print("\n  New York forecast 2026-2035:")
for _, r in ny_poly_fc.iterrows():
    print(f"    {int(r.Year)}: ${r.Forecast/1e9:.2f}B  [${r.Lower_95/1e9:.2f}B - ${r.Upper_95/1e9:.2f}B]")

print("\n✅ Model 2 complete")

MODEL 2: POLYNOMIAL REGRESSION (DEGREE 2)

  National:
    Polynomial Regression:
      MAE:  $120.44B
      RMSE: $126.09B
      MAPE: 12.8%

  New York:
    Polynomial Regression:
      MAE:  $29.88B
      RMSE: $31.92B
      MAPE: 30.8%

  National forecast 2026-2035:
    2026: $892.50B  [$835.19B - $949.80B]
    2027: $928.94B  [$871.63B - $986.24B]
    2028: $965.89B  [$908.59B - $1023.19B]
    2029: $1003.35B  [$946.05B - $1060.66B]
    2030: $1041.33B  [$984.02B - $1098.63B]
    2031: $1079.81B  [$1022.51B - $1137.12B]
    2032: $1118.81B  [$1061.51B - $1176.12B]
    2033: $1158.32B  [$1101.02B - $1215.62B]
    2034: $1198.34B  [$1141.04B - $1255.64B]
    2035: $1238.87B  [$1181.57B - $1296.18B]

  New York forecast 2026-2035:
    2026: $53.56B  [$35.68B - $71.44B]
    2027: $47.29B  [$29.41B - $65.17B]
    2028: $40.11B  [$22.23B - $57.99B]
    2029: $32.02B  [$14.14B - $49.90B]
    2030: $23.00B  [$5.12B - $40.88B]
    2031: $13.08B  [$-4.80B - $30.96B]
    2032: $2.23B  [$-15

In [16]:
# ============================================================
# MODEL 3: HOLT'S EXPONENTIAL SMOOTHING
# ============================================================

def holt_smooth(y, alpha, beta):
    n    = len(y)
    L    = np.zeros(n)
    T    = np.zeros(n)
    L[0] = y[0]
    T[0] = y[1] - y[0]
    for t in range(1, n):
        L[t] = alpha * y[t] + (1 - alpha) * (L[t-1] + T[t-1])
        T[t] = beta  * (L[t] - L[t-1]) + (1 - beta) * T[t-1]
    return L, T

def holt_forecast(L, T, h):
    return np.array([L[-1] + i * T[-1] for i in range(1, h+1)])

def fit_holt(ts_df, train_end, future_years):
    train, test = train_test_split_ts(ts_df, train_end)
    y_train  = train['Expenditure'].values.astype(float)
    y_test   = test['Expenditure'].values.astype(float)
    n_test   = len(test)
    n_future = len(future_years)

    def sse(params):
        a, b = params
        if not (0 < a < 1) or not (0 < b < 1):
            return 1e20
        L, T = holt_smooth(y_train, a, b)
        fitted = np.array([L[t-1] + T[t-1] for t in range(1, len(y_train))])
        return np.sum((y_train[1:] - fitted) ** 2)

    res = minimize(sse, x0=[0.5, 0.1], method='Nelder-Mead',
                   options={'xatol':1e-8, 'fatol':1e-8, 'maxiter':5000})
    alpha_opt = np.clip(res.x[0], 1e-4, 1-1e-4)
    beta_opt  = np.clip(res.x[1], 1e-4, 1-1e-4)

    y_all = y_train.copy()
    test_pred = []
    for _ in range(n_test):
        L_t, T_t = holt_smooth(y_all, alpha_opt, beta_opt)
        next_val = L_t[-1] + T_t[-1]
        test_pred.append(next_val)
        y_all = np.append(y_all, next_val)

    test_pred = np.array(test_pred)
    metrics   = compute_metrics(y_test, test_pred,
                f"Holt's Smoothing (alpha={alpha_opt:.2f}, beta={beta_opt:.2f})")

    y_full   = ts_df['Expenditure'].values.astype(float)
    L_a, T_a = holt_smooth(y_full, alpha_opt, beta_opt)
    fc_vals  = holt_forecast(L_a, T_a, n_future)

    L_tr, T_tr = holt_smooth(y_train, alpha_opt, beta_opt)
    fitted_in  = np.array([L_tr[t-1] + T_tr[t-1] for t in range(1, len(y_train))])
    sigma      = np.std(y_train[1:] - fitted_in)
    n_tr       = len(y_train)
    t_crit     = stats.t.ppf(0.975, df=max(n_tr-2, 1))
    ci_widths  = np.array([t_crit * sigma * np.sqrt(h) for h in range(1, n_future+1)])

    forecast_df = pd.DataFrame({
        'Year':     future_years,
        'Forecast': fc_vals,
        'Lower_95': fc_vals - ci_widths,
        'Upper_95': fc_vals + ci_widths,
    })
    return alpha_opt, beta_opt, metrics, forecast_df, test_pred

print("=" * 45)
print("MODEL 3: HOLT'S EXPONENTIAL SMOOTHING")
print("=" * 45)

print("\n  National:")
nat_hw_alpha, nat_hw_beta, nat_hw_metrics, nat_hw_fc, nat_hw_test_pred = fit_holt(national_ts, TRAIN_END, FUTURE_YEARS)

print("\n  New York:")
ny_hw_alpha, ny_hw_beta, ny_hw_metrics, ny_hw_fc, ny_hw_test_pred = fit_holt(ny_ts, TRAIN_END, FUTURE_YEARS)

print("\n  National forecast 2026-2035:")
for _, r in nat_hw_fc.iterrows():
    print(f"    {int(r.Year)}: ${r.Forecast/1e9:.2f}B  [${r.Lower_95/1e9:.2f}B - ${r.Upper_95/1e9:.2f}B]")

print("\n  New York forecast 2026-2035:")
for _, r in ny_hw_fc.iterrows():
    print(f"    {int(r.Year)}: ${r.Forecast/1e9:.2f}B  [${r.Lower_95/1e9:.2f}B - ${r.Upper_95/1e9:.2f}B]")

print("\n✅ Model 3 complete")

MODEL 3: HOLT'S EXPONENTIAL SMOOTHING

  National:
    Holt's Smoothing (alpha=0.64, beta=0.00):
      MAE:  $107.64B
      RMSE: $113.84B
      MAPE: 11.4%

  New York:
    Holt's Smoothing (alpha=0.63, beta=0.05):
      MAE:  $20.66B
      RMSE: $21.76B
      MAPE: 21.4%

  National forecast 2026-2035:
    2026: $1044.02B  [$990.23B - $1097.82B]
    2027: $1079.20B  [$1003.11B - $1155.28B]
    2028: $1114.37B  [$1021.18B - $1207.55B]
    2029: $1149.54B  [$1041.94B - $1257.14B]
    2030: $1184.71B  [$1064.41B - $1305.01B]
    2031: $1219.88B  [$1088.10B - $1351.66B]
    2032: $1255.05B  [$1112.71B - $1397.39B]
    2033: $1290.22B  [$1138.06B - $1442.39B]
    2034: $1325.39B  [$1164.00B - $1486.79B]
    2035: $1360.56B  [$1190.44B - $1530.69B]

  New York forecast 2026-2035:
    2026: $102.45B  [$82.45B - $122.46B]
    2027: $104.32B  [$76.02B - $132.61B]
    2028: $106.18B  [$71.53B - $140.83B]
    2029: $108.04B  [$68.03B - $148.06B]
    2030: $109.91B  [$65.17B - $154.64B]
    2031

In [17]:
# ============================================================
# MODEL 4: ARIMA(1,1,1)
# ============================================================

def arima_111(ts_df, train_end, future_years):
    train, test = train_test_split_ts(ts_df, train_end)
    y_train  = train['Expenditure'].values.astype(float)
    y_test   = test['Expenditure'].values.astype(float)
    n_future = len(future_years)

    # d=1: work on first differences
    dy = np.diff(y_train)
    n  = len(dy)

    # Estimate AR(1) and MA(1) parameters
    # by minimising conditional sum of squares
    def css(params):
        phi, theta = params
        if abs(phi) >= 1 or abs(theta) >= 1:
            return 1e20
        e = np.zeros(n)
        for t in range(1, n):
            e[t] = dy[t] - phi * dy[t-1] - theta * e[t-1]
        return np.sum(e[1:] ** 2)

    res = minimize(css, x0=[0.3, 0.3], method='Nelder-Mead',
                   options={'xatol':1e-8, 'fatol':1e-8, 'maxiter':5000})
    phi   = np.clip(res.x[0], -0.99, 0.99)
    theta = np.clip(res.x[1], -0.99, 0.99)

    # Residuals on training set
    e_train = np.zeros(n)
    for t in range(1, n):
        e_train[t] = dy[t] - phi * dy[t-1] - theta * e_train[t-1]
    sigma = np.std(e_train[1:])

    # Rolling 1-step ahead test predictions
    y_full = y_train.copy()
    dy_all = dy.copy()
    e_all  = e_train.copy()
    test_pred = []

    for _ in range(len(y_test)):
        pred_dy = phi * dy_all[-1] + theta * e_all[-1]
        pred_y  = y_full[-1] + pred_dy
        test_pred.append(pred_y)
        dy_all = np.append(dy_all, pred_dy)
        e_all  = np.append(e_all,  0.0)
        y_full = np.append(y_full, pred_y)

    test_pred = np.array(test_pred)
    metrics   = compute_metrics(y_test, test_pred,
                f"ARIMA(1,1,1) (phi={phi:.2f}, theta={theta:.2f})")

    # Future forecast using full dataset
    y_all2  = ts_df['Expenditure'].values.astype(float)
    dy_all2 = np.diff(y_all2)
    e_all2  = np.zeros(len(dy_all2))
    for t in range(1, len(dy_all2)):
        e_all2[t] = dy_all2[t] - phi * dy_all2[t-1] - theta * e_all2[t-1]

    sigma2 = np.std(e_all2[1:])
    t_crit = stats.t.ppf(0.975, df=max(len(y_all2)-3, 1))

    fc_vals   = []
    ci_widths = []
    last_y    = y_all2[-1]
    last_dy   = dy_all2[-1]
    last_e    = e_all2[-1]
    pred_dy   = 0.0

    for h in range(1, n_future+1):
        if h == 1:
            pred_dy = phi * last_dy + theta * last_e
        else:
            pred_dy = phi * pred_dy
        pred_y = last_y + pred_dy
        fc_vals.append(pred_y)
        ci_widths.append(t_crit * sigma2 * np.sqrt(h))
        last_y = pred_y

    fc_vals   = np.array(fc_vals)
    ci_widths = np.array(ci_widths)

    forecast_df = pd.DataFrame({
        'Year':     future_years,
        'Forecast': fc_vals,
        'Lower_95': fc_vals - ci_widths,
        'Upper_95': fc_vals + ci_widths,
    })
    return phi, theta, metrics, forecast_df, test_pred

print("=" * 45)
print("MODEL 4: ARIMA(1,1,1)")
print("=" * 45)

print("\n  National:")
nat_ar_phi, nat_ar_theta, nat_ar_metrics, nat_ar_fc, nat_ar_test_pred = arima_111(national_ts, TRAIN_END, FUTURE_YEARS)

print("\n  New York:")
ny_ar_phi, ny_ar_theta, ny_ar_metrics, ny_ar_fc, ny_ar_test_pred = arima_111(ny_ts, TRAIN_END, FUTURE_YEARS)

print("\n  National forecast 2026-2035:")
for _, r in nat_ar_fc.iterrows():
    print(f"    {int(r.Year)}: ${r.Forecast/1e9:.2f}B  [${r.Lower_95/1e9:.2f}B - ${r.Upper_95/1e9:.2f}B]")

print("\n  New York forecast 2026-2035:")
for _, r in ny_ar_fc.iterrows():
    print(f"    {int(r.Year)}: ${r.Forecast/1e9:.2f}B  [${r.Lower_95/1e9:.2f}B - ${r.Upper_95/1e9:.2f}B]")

print("\n✅ Model 4 complete")

MODEL 4: ARIMA(1,1,1)

  National:
    ARIMA(1,1,1) (phi=0.99, theta=-0.04):
      MAE:  $33.74B
      RMSE: $35.51B
      MAPE: 3.6%

  New York:
    ARIMA(1,1,1) (phi=-0.80, theta=0.99):
      MAE:  $22.05B
      RMSE: $23.19B
      MAPE: 22.8%

  National forecast 2026-2035:
    2026: $1112.55B  [$1055.87B - $1169.24B]
    2027: $1192.59B  [$1112.43B - $1272.75B]
    2028: $1271.83B  [$1173.65B - $1370.01B]
    2029: $1350.27B  [$1236.90B - $1463.64B]
    2030: $1427.93B  [$1301.18B - $1554.68B]
    2031: $1504.82B  [$1365.97B - $1643.67B]
    2032: $1580.93B  [$1430.96B - $1730.91B]
    2033: $1656.29B  [$1495.96B - $1816.62B]
    2034: $1730.89B  [$1560.84B - $1900.94B]
    2035: $1804.74B  [$1625.49B - $1984.00B]

  New York forecast 2026-2035:
    2026: $102.09B  [$85.03B - $119.15B]
    2027: $102.58B  [$78.46B - $126.70B]
    2028: $102.19B  [$72.64B - $131.73B]
    2029: $102.50B  [$68.39B - $136.62B]
    2030: $102.25B  [$64.11B - $140.39B]
    2031: $102.45B  [$60.67B - $14

In [18]:
# ============================================================
# STEP 6: MODEL COMPARISON & BEST MODEL SELECTION
# ============================================================

def make_comparison(metrics_lin, metrics_poly, metrics_hw, metrics_ar, label):
    rows = [
        {'Model': 'Linear Regression',            **metrics_lin},
        {'Model': 'Polynomial Regression (d=2)',   **metrics_poly},
        {'Model': "Holt's Exponential Smoothing",  **metrics_hw},
        {'Model': 'ARIMA(1,1,1)',                  **metrics_ar},
    ]
    comp = pd.DataFrame(rows)
    comp['MAE (Billions)']  = comp['MAE']  / 1e9
    comp['RMSE (Billions)'] = comp['RMSE'] / 1e9
    comp['MAPE (%)']        = comp['MAPE']
    comp = comp[['Model','MAE (Billions)','RMSE (Billions)','MAPE (%)']]
    comp = comp.sort_values('MAPE (%)').reset_index(drop=True)
    comp.insert(0, 'Rank', range(1, len(comp)+1))
    
    print(f"\n  {label} - Test Period FY 2022-2025:")
    print(f"  {'Rank':<6}{'Model':<35}{'MAE':>12}{'RMSE':>12}{'MAPE':>10}")
    print(f"  {'-'*75}")
    for _, r in comp.iterrows():
        star = " <-- BEST" if r['Rank'] == 1 else ""
        print(f"  {int(r['Rank']):<6}{r['Model']:<35}"
              f"  ${r['MAE (Billions)']:>8.2f}B"
              f"  ${r['RMSE (Billions)']:>8.2f}B"
              f"  {r['MAPE (%)']:>6.1f}%{star}")
    
    best = comp.iloc[0]['Model']
    print(f"\n  Best model for {label}: {best}")
    return comp, best

print("=" * 55)
print("MODEL COMPARISON SUMMARY")
print("=" * 55)

nat_comp, nat_best = make_comparison(
    nat_lin_metrics, nat_poly_metrics,
    nat_hw_metrics,  nat_ar_metrics,
    "NATIONAL"
)

ny_comp, ny_best = make_comparison(
    ny_lin_metrics, ny_poly_metrics,
    ny_hw_metrics,  ny_ar_metrics,
    "NEW YORK"
)

# Select best forecasts
model_map_nat = {
    'Linear Regression':            nat_lin_fc,
    'Polynomial Regression (d=2)':  nat_poly_fc,
    "Holt's Exponential Smoothing": nat_hw_fc,
    'ARIMA(1,1,1)':                 nat_ar_fc,
}
model_map_ny = {
    'Linear Regression':            ny_lin_fc,
    'Polynomial Regression (d=2)':  ny_poly_fc,
    "Holt's Exponential Smoothing": ny_hw_fc,
    'ARIMA(1,1,1)':                 ny_ar_fc,
}

nat_best_fc = model_map_nat[nat_best]
ny_best_fc  = model_map_ny[ny_best]

print("\n" + "=" * 55)
print("FINAL FORECAST SUMMARY (BEST MODELS)")
print("=" * 55)

print(f"\n  National - {nat_best}")
print(f"    2026: ${nat_best_fc.iloc[0]['Forecast']/1e9:.2f}B")
print(f"    2035: ${nat_best_fc.iloc[-1]['Forecast']/1e9:.2f}B")

print(f"\n  New York - {ny_best}")
print(f"    2026: ${ny_best_fc.iloc[0]['Forecast']/1e9:.2f}B")
print(f"    2035: ${ny_best_fc.iloc[-1]['Forecast']/1e9:.2f}B")

print("\n✅ Model comparison complete")

MODEL COMPARISON SUMMARY

  NATIONAL - Test Period FY 2022-2025:
  Rank  Model                                       MAE        RMSE      MAPE
  ---------------------------------------------------------------------------
  1     ARIMA(1,1,1)                         $   33.74B  $   35.51B     3.6% <-- BEST
  2     Holt's Exponential Smoothing         $  107.64B  $  113.84B    11.4%
  3     Polynomial Regression (d=2)          $  120.44B  $  126.09B    12.8%
  4     Linear Regression                    $  129.85B  $  136.18B    13.8%

  Best model for NATIONAL: ARIMA(1,1,1)

  NEW YORK - Test Period FY 2022-2025:
  Rank  Model                                       MAE        RMSE      MAPE
  ---------------------------------------------------------------------------
  1     Linear Regression                    $   13.00B  $   13.90B    13.4% <-- BEST
  2     Holt's Exponential Smoothing         $   20.66B  $   21.76B    21.4%
  3     ARIMA(1,1,1)                         $   22.05B  $   2

In [19]:
# ============================================================
# STEP 7: VISUALIZATIONS
# ============================================================

# ── Chart 1: Best model forecast - National ──────────────────

def plot_forecast(ts_df, fc_df, model_name, title, color_hist, fname):
    fig, ax = plt.subplots(figsize=(12, 6))

    # Historical data
    ax.plot(ts_df['Year'], ts_df['Expenditure']/1e9,
            color=color_hist, marker='o', linewidth=2.5,
            markersize=8, label='Historical', zorder=3)

    # Confidence interval band
    ax.fill_between(fc_df['Year'],
                    fc_df['Lower_95']/1e9,
                    fc_df['Upper_95']/1e9,
                    alpha=0.20, color='#ff7f0e',
                    label='95% Confidence Interval')

    # Forecast line
    ax.plot(fc_df['Year'], fc_df['Forecast']/1e9,
            color='#ff7f0e', marker='o', linewidth=2.5,
            markersize=8, linestyle='--',
            label=f'Forecast ({model_name})', zorder=3)

    # Vertical line separating historical from forecast
    ax.axvline(x=2025.5, color='gray', linestyle=':',
               linewidth=1.2, alpha=0.7)
    ax.text(2025.7, ax.get_ylim()[1]*0.95,
            'Forecast', color='gray', fontsize=9)

    # Annotate first and last forecast values
    ax.annotate(f"${fc_df.iloc[0]['Forecast']/1e9:.1f}B",
                xy=(fc_df.iloc[0]['Year'],
                    fc_df.iloc[0]['Forecast']/1e9),
                xytext=(8, 8), textcoords='offset points',
                fontsize=9, color='#ff7f0e')
    ax.annotate(f"${fc_df.iloc[-1]['Forecast']/1e9:.1f}B",
                xy=(fc_df.iloc[-1]['Year'],
                    fc_df.iloc[-1]['Forecast']/1e9),
                xytext=(8, -14), textcoords='offset points',
                fontsize=9, color='#ff7f0e')

    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Fiscal Year', fontsize=11)
    ax.set_ylabel('Expenditure (Billions USD)', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{fname}", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {fname}")

plot_forecast(
    national_ts, nat_best_fc, nat_best,
    f"National Medicaid Expenditure Forecast (2013-2035)\nBest Model: {nat_best}",
    '#1f77b4', "03_Forecast_National.png"
)

plot_forecast(
    ny_ts, ny_best_fc, ny_best,
    f"New York Medicaid Expenditure Forecast (2013-2035)\nBest Model: {ny_best}",
    '#d62728', "04_Forecast_NewYork.png"
)

print("\n✅ Forecast charts saved")

  Saved: 03_Forecast_National.png
  Saved: 04_Forecast_NewYork.png

✅ Forecast charts saved


In [20]:
# ── Chart 2: All models comparison ───────────────────────────

def plot_all_models(ts_df, fc_dict, title, color_hist, fname):
    fig, ax = plt.subplots(figsize=(13, 7))
    palette = ['#ff7f0e', '#2ca02c', '#9467bd', '#8c564b']
    markers = ['o', 's', 'D', '^']

    # Historical
    ax.plot(ts_df['Year'], ts_df['Expenditure']/1e9,
            color=color_hist, marker='o', linewidth=2.5,
            markersize=8, label='Historical', zorder=4)

    # Vertical divider
    ax.axvline(x=2025.5, color='gray', linestyle=':',
               linewidth=1, alpha=0.7)

    # Each model forecast
    for (name, fc_df), col, mk in zip(fc_dict.items(), palette, markers):
        ax.plot(fc_df['Year'], fc_df['Forecast']/1e9,
                color=col, marker=mk, linewidth=1.8,
                markersize=7, linestyle='--',
                label=name, alpha=0.85)
        ax.fill_between(fc_df['Year'],
                        fc_df['Lower_95']/1e9,
                        fc_df['Upper_95']/1e9,
                        alpha=0.06, color=col)

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Fiscal Year')
    ax.set_ylabel('Expenditure (Billions USD)')
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{fname}", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {fname}")

nat_fc_dict = {
    'Linear Regression':  nat_lin_fc,
    'Polynomial (d=2)':   nat_poly_fc,
    "Holt's Smoothing":   nat_hw_fc,
    'ARIMA(1,1,1)':       nat_ar_fc,
}
ny_fc_dict = {
    'Linear Regression':  ny_lin_fc,
    'Polynomial (d=2)':   ny_poly_fc,
    "Holt's Smoothing":   ny_hw_fc,
    'ARIMA(1,1,1)':       ny_ar_fc,
}

plot_all_models(
    national_ts, nat_fc_dict,
    "National Medicaid Expenditure - All Models Comparison (2026-2035)",
    '#1f77b4', "05_AllModels_National.png"
)

plot_all_models(
    ny_ts, ny_fc_dict,
    "New York Medicaid Expenditure - All Models Comparison (2026-2035)",
    '#d62728', "06_AllModels_NewYork.png"
)

print("\n✅ All models comparison charts saved")

  Saved: 05_AllModels_National.png
  Saved: 06_AllModels_NewYork.png

✅ All models comparison charts saved


In [22]:
# ============================================================
# EXPORT RESULTS TO EXCEL AND CSV
# ============================================================

def prep_export(fc_df, label, model_name):
    out = fc_df.copy()
    out.columns = ['Fiscal_Year', 'Forecast_USD',
                   'Lower_95_CI_USD', 'Upper_95_CI_USD']
    out['Scope']                = label
    out['Best_Model']           = model_name
    out['Forecast_Billions']    = (out['Forecast_USD']    / 1e9).round(2)
    out['Lower_95_CI_Billions'] = (out['Lower_95_CI_USD'] / 1e9).round(2)
    out['Upper_95_CI_Billions'] = (out['Upper_95_CI_USD'] / 1e9).round(2)
    return out[['Fiscal_Year', 'Scope', 'Best_Model',
                'Forecast_Billions', 'Lower_95_CI_Billions',
                'Upper_95_CI_Billions', 'Forecast_USD',
                'Lower_95_CI_USD', 'Upper_95_CI_USD']]

nat_export = prep_export(nat_best_fc, 'National', nat_best)
ny_export  = prep_export(ny_best_fc,  'New York', ny_best)
combined   = pd.concat([nat_export, ny_export], ignore_index=True)

excel_path = f"{OUT_DIR}/Medicaid_Expenditure_Forecast_2026_2035.xlsx"

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    combined.to_excel(writer, sheet_name='Combined_Forecast', index=False)
    nat_export.to_excel(writer, sheet_name='National_Forecast', index=False)
    ny_export.to_excel(writer, sheet_name='NewYork_Forecast', index=False)
    nat_comp.to_excel(writer, sheet_name='National_Model_Comparison', index=False)
    ny_comp.to_excel(writer, sheet_name='NewYork_Model_Comparison', index=False)
    national_ts.to_excel(writer, sheet_name='Historical_National', index=False)
    ny_ts.to_excel(writer, sheet_name='Historical_NewYork', index=False)

print("Excel saved: Medicaid_Expenditure_Forecast_2026_2035.xlsx")
print("  Sheet 1: Combined_Forecast")
print("  Sheet 2: National_Forecast")
print("  Sheet 3: NewYork_Forecast")
print("  Sheet 4: National_Model_Comparison")
print("  Sheet 5: NewYork_Model_Comparison")
print("  Sheet 6: Historical_National")
print("  Sheet 7: Historical_NewYork")

nat_export.to_csv(f"{OUT_DIR}/National_Forecast_2026_2035.csv", index=False)
ny_export.to_csv(f"{OUT_DIR}/NewYork_Forecast_2026_2035.csv", index=False)

print("\nCSV saved: National_Forecast_2026_2035.csv")
print("CSV saved: NewYork_Forecast_2026_2035.csv")

print("\nPreview of combined forecast table:")
print(combined[['Fiscal_Year', 'Scope', 'Best_Model',
                'Forecast_Billions', 'Lower_95_CI_Billions',
                'Upper_95_CI_Billions']].to_string(index=False))

print("\n✅ All files exported successfully")

Excel saved: Medicaid_Expenditure_Forecast_2026_2035.xlsx
  Sheet 1: Combined_Forecast
  Sheet 2: National_Forecast
  Sheet 3: NewYork_Forecast
  Sheet 4: National_Model_Comparison
  Sheet 5: NewYork_Model_Comparison
  Sheet 6: Historical_National
  Sheet 7: Historical_NewYork

CSV saved: National_Forecast_2026_2035.csv
CSV saved: NewYork_Forecast_2026_2035.csv

Preview of combined forecast table:
 Fiscal_Year    Scope        Best_Model  Forecast_Billions  Lower_95_CI_Billions  Upper_95_CI_Billions
        2026 National      ARIMA(1,1,1)            1112.55               1055.87               1169.24
        2027 National      ARIMA(1,1,1)            1192.59               1112.43               1272.75
        2028 National      ARIMA(1,1,1)            1271.83               1173.65               1370.01
        2029 National      ARIMA(1,1,1)            1350.27               1236.90               1463.64
        2030 National      ARIMA(1,1,1)            1427.93               1301.18    

In [23]:
# ============================================================
# INTERACTIVE HTML DASHBOARD
# ============================================================

def series_to_js(years, values):
    return str([(int(y), round(v/1e9, 3)) for y, v in zip(years, values)])

def ci_to_js(years, lower, upper):
    return str([(int(y), round(l/1e9, 3), round(u/1e9, 3))
                for y, l, u in zip(years, lower, upper)])

# Convert all data to JavaScript format
nat_hist_js    = series_to_js(national_ts['Year'], national_ts['Expenditure'])
ny_hist_js     = series_to_js(ny_ts['Year'],       ny_ts['Expenditure'])

nat_lin_js     = series_to_js(nat_lin_fc['Year'],  nat_lin_fc['Forecast'])
nat_poly_js    = series_to_js(nat_poly_fc['Year'], nat_poly_fc['Forecast'])
nat_hw_js      = series_to_js(nat_hw_fc['Year'],   nat_hw_fc['Forecast'])
nat_ar_js      = series_to_js(nat_ar_fc['Year'],   nat_ar_fc['Forecast'])
nat_best_js    = series_to_js(nat_best_fc['Year'], nat_best_fc['Forecast'])
nat_best_ci_js = ci_to_js(nat_best_fc['Year'],
                           nat_best_fc['Lower_95'],
                           nat_best_fc['Upper_95'])

ny_lin_js      = series_to_js(ny_lin_fc['Year'],   ny_lin_fc['Forecast'])
ny_poly_js     = series_to_js(ny_poly_fc['Year'],  ny_poly_fc['Forecast'])
ny_hw_js       = series_to_js(ny_hw_fc['Year'],    ny_hw_fc['Forecast'])
ny_ar_js       = series_to_js(ny_ar_fc['Year'],    ny_ar_fc['Forecast'])
ny_best_js     = series_to_js(ny_best_fc['Year'],  ny_best_fc['Forecast'])
ny_best_ci_js  = ci_to_js(ny_best_fc['Year'],
                           ny_best_fc['Lower_95'],
                           ny_best_fc['Upper_95'])

# Model comparison table rows
def comp_to_html(comp_df):
    rows = ""
    for _, r in comp_df.iterrows():
        bold = ' style="font-weight:700;background:#fffde7;"' if r['Rank']==1 else ''
        rows += (f'<tr{bold}><td>{int(r["Rank"])}</td>'
                 f'<td>{r["Model"]}</td>'
                 f'<td>${r["MAE (Billions)"]:.2f}B</td>'
                 f'<td>${r["RMSE (Billions)"]:.2f}B</td>'
                 f'<td>{r["MAPE (%)"]:.1f}%</td></tr>')
    return rows

# Forecast table rows
def fc_to_html(fc_df):
    rows = ""
    for _, r in fc_df.iterrows():
        rows += (f'<tr><td>{int(r["Fiscal_Year"])}</td>'
                 f'<td>${r["Forecast_Billions"]:.2f}B</td>'
                 f'<td>${r["Lower_95_CI_Billions"]:.2f}B</td>'
                 f'<td>${r["Upper_95_CI_Billions"]:.2f}B</td></tr>')
    return rows

nat_comp_rows = comp_to_html(nat_comp)
ny_comp_rows  = comp_to_html(ny_comp)
nat_fc_rows   = fc_to_html(nat_export)
ny_fc_rows    = fc_to_html(ny_export)

print("Data prepared for dashboard...")
print("Building HTML...")

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>Medicaid Expenditure Forecast 2026-2035</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
  *{{box-sizing:border-box;margin:0;padding:0;}}
  body{{font-family:'Segoe UI',Arial,sans-serif;background:#f4f6fa;color:#222;}}
  header{{background:linear-gradient(135deg,#1565c0,#0d47a1);color:#fff;padding:28px 36px;}}
  header h1{{font-size:1.7rem;font-weight:700;}}
  header p{{margin-top:6px;opacity:0.85;font-size:0.95rem;}}
  .container{{max-width:1180px;margin:0 auto;padding:28px 24px;}}
  .tabs{{display:flex;gap:8px;margin-bottom:20px;flex-wrap:wrap;}}
  .tab-btn{{padding:9px 22px;border:none;border-radius:6px;cursor:pointer;
    font-size:0.95rem;font-weight:600;background:#e3eaf6;color:#1565c0;transition:all .2s;}}
  .tab-btn.active{{background:#1565c0;color:#fff;}}
  .tab-btn:hover:not(.active){{background:#c5d8f5;}}
  .tab-panel{{display:none;}}
  .tab-panel.active{{display:block;}}
  .card{{background:#fff;border-radius:10px;box-shadow:0 2px 12px rgba(0,0,0,.08);
    padding:24px;margin-bottom:24px;}}
  .card h2{{font-size:1.1rem;color:#1565c0;margin-bottom:16px;
    border-bottom:2px solid #e3eaf6;padding-bottom:8px;}}
  .chart-wrap{{position:relative;height:380px;}}
  .subtabs{{display:flex;gap:6px;margin-bottom:14px;flex-wrap:wrap;}}
  .subtab-btn{{padding:6px 16px;border:1px solid #c5d8f5;border-radius:20px;
    cursor:pointer;font-size:0.87rem;background:#f0f5ff;color:#1565c0;transition:all .15s;}}
  .subtab-btn.active{{background:#1565c0;color:#fff;border-color:#1565c0;}}
  table{{width:100%;border-collapse:collapse;font-size:0.9rem;}}
  th{{background:#1565c0;color:#fff;padding:9px 12px;text-align:left;}}
  td{{padding:8px 12px;border-bottom:1px solid #eee;}}
  tr:hover td{{background:#f0f5ff;}}
  .grid-2{{display:grid;grid-template-columns:1fr 1fr;gap:20px;}}
  .badge{{display:inline-block;padding:2px 10px;border-radius:12px;
    font-size:0.8rem;font-weight:700;background:#e8f5e9;color:#2e7d32;}}
  footer{{text-align:center;padding:20px;color:#888;font-size:0.82rem;}}
  @media(max-width:700px){{.grid-2{{grid-template-columns:1fr;}}}}
</style>
</head>
<body>
<header>
  <h1>Medicaid Expenditure Projection Model</h1>
  <p>Historical Analysis (FY 2013-2025) and 10-Year Forecast (2026-2035)
     | National and New York State
     | Author: Asmita Thapa</p>
</header>

<div class="container">
  <div class="tabs">
    <button class="tab-btn active" onclick="switchTab('national',this)">National</button>
    <button class="tab-btn"        onclick="switchTab('newyork',this)">New York State</button>
    <button class="tab-btn"        onclick="switchTab('compare',this)">Model Comparison</button>
    <button class="tab-btn"        onclick="switchTab('tables',this)">Forecast Tables</button>
  </div>

  <!-- NATIONAL TAB -->
  <div id="tab-national" class="tab-panel active">
    <div class="card">
      <h2>National Medicaid Expenditure - Best Model Forecast
        <span class="badge">{nat_best}</span></h2>
      <div class="chart-wrap"><canvas id="natBestChart"></canvas></div>
    </div>
    <div class="card">
      <h2>National - All Models Comparison</h2>
      <div class="subtabs">
        <button class="subtab-btn active" onclick="toggleNat('all',this)">All Models</button>
        <button class="subtab-btn" onclick="toggleNat('linear',this)">Linear</button>
        <button class="subtab-btn" onclick="toggleNat('poly',this)">Polynomial</button>
        <button class="subtab-btn" onclick="toggleNat('holt',this)">Holt's</button>
        <button class="subtab-btn" onclick="toggleNat('arima',this)">ARIMA</button>
      </div>
      <div class="chart-wrap"><canvas id="natAllChart"></canvas></div>
    </div>
  </div>

  <!-- NEW YORK TAB -->
  <div id="tab-newyork" class="tab-panel">
    <div class="card">
      <h2>New York State Medicaid Expenditure - Best Model Forecast
        <span class="badge">{ny_best}</span></h2>
      <div class="chart-wrap"><canvas id="nyBestChart"></canvas></div>
    </div>
    <div class="card">
      <h2>New York - All Models Comparison</h2>
      <div class="subtabs">
        <button class="subtab-btn active" onclick="toggleNy('all',this)">All Models</button>
        <button class="subtab-btn" onclick="toggleNy('linear',this)">Linear</button>
        <button class="subtab-btn" onclick="toggleNy('poly',this)">Polynomial</button>
        <button class="subtab-btn" onclick="toggleNy('holt',this)">Holt's</button>
        <button class="subtab-btn" onclick="toggleNy('arima',this)">ARIMA</button>
      </div>
      <div class="chart-wrap"><canvas id="nyAllChart"></canvas></div>
    </div>
  </div>

  <!-- MODEL COMPARISON TAB -->
  <div id="tab-compare" class="tab-panel">
    <div class="grid-2">
      <div class="card">
        <h2>National - Model Performance (Test: FY 2022-2025)</h2>
        <table>
          <thead><tr><th>Rank</th><th>Model</th>
            <th>MAE</th><th>RMSE</th><th>MAPE</th></tr></thead>
          <tbody>{nat_comp_rows}</tbody>
        </table>
      </div>
      <div class="card">
        <h2>New York - Model Performance (Test: FY 2022-2025)</h2>
        <table>
          <thead><tr><th>Rank</th><th>Model</th>
            <th>MAE</th><th>RMSE</th><th>MAPE</th></tr></thead>
          <tbody>{ny_comp_rows}</tbody>
        </table>
      </div>
    </div>
    <div class="card">
      <h2>Methodology Notes</h2>
      <p style="line-height:1.8;font-size:0.93rem;">
        <strong>Train/Test Split:</strong> Models trained on FY 2013-2021,
        evaluated on FY 2022-2025.<br>
        <strong>Linear Regression:</strong> OLS fit on Year as predictor.
        Prediction intervals widen using t-distribution leverage formula.<br>
        <strong>Polynomial Regression (d=2):</strong> Quadratic OLS.
        Captures non-linear acceleration in spending.<br>
        <strong>Holt's Exponential Smoothing:</strong> Level (alpha) and trend
        (beta) parameters optimised by minimising SSE. CI widens as sqrt(h).<br>
        <strong>ARIMA(1,1,1):</strong> AR and MA parameters estimated by
        conditional least squares on first differences.<br>
        <strong>Best model selected</strong> by lowest MAPE on hold-out test set.
      </p>
    </div>
  </div>

  <!-- FORECAST TABLES TAB -->
  <div id="tab-tables" class="tab-panel">
    <div class="grid-2">
      <div class="card">
        <h2>National Forecast 2026-2035
          <span class="badge">{nat_best}</span></h2>
        <table>
          <thead><tr><th>Year</th><th>Forecast</th>
            <th>Lower 95% CI</th><th>Upper 95% CI</th></tr></thead>
          <tbody>{nat_fc_rows}</tbody>
        </table>
      </div>
      <div class="card">
        <h2>New York Forecast 2026-2035
          <span class="badge">{ny_best}</span></h2>
        <table>
          <thead><tr><th>Year</th><th>Forecast</th>
            <th>Lower 95% CI</th><th>Upper 95% CI</th></tr></thead>
          <tbody>{ny_fc_rows}</tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<footer>Medicaid Expenditure Projection Model | FY 2013-2025 Data |
  Models: Linear Regression, Polynomial Regression,
  Holt's Exponential Smoothing, ARIMA(1,1,1) | Author: Asmita Thapa</footer>

<script>
const natHist   = {nat_hist_js};
const nyHist    = {ny_hist_js};
const natLinFc  = {nat_lin_js};
const natPolyFc = {nat_poly_js};
const natHwFc   = {nat_hw_js};
const natArFc   = {nat_ar_js};
const natBestFc = {nat_best_js};
const natBestCI = {nat_best_ci_js};
const nyLinFc   = {ny_lin_js};
const nyPolyFc  = {ny_poly_js};
const nyHwFc    = {ny_hw_js};
const nyArFc    = {ny_ar_js};
const nyBestFc  = {ny_best_js};
const nyBestCI  = {ny_best_ci_js};

function mkDS(label, pts, color, dash) {{
  return {{
    label,
    data: pts.map(p => ({{x:p[0], y:p[1]}})),
    borderColor: color,
    backgroundColor: color,
    borderWidth: 2.2,
    borderDash: dash || [],
    pointRadius: 5,
    tension: 0.25
  }};
}}

function mkCI(pts, color) {{
  return [
    {{ label:'Upper CI',
       data: pts.map(p => ({{x:p[0], y:p[2]}})),
       borderColor:'transparent',
       backgroundColor: color+'28',
       borderWidth:0, pointRadius:0,
       fill:'+1', tension:0.25 }},
    {{ label:'Lower CI',
       data: pts.map(p => ({{x:p[0], y:p[1]}})),
       borderColor:'transparent',
       backgroundColor: color+'28',
       borderWidth:0, pointRadius:0,
       fill:false, tension:0.25 }}
  ];
}}

const OPT = {{
  responsive:true,
  maintainAspectRatio:false,
  interaction:{{mode:'index', intersect:false}},
  plugins:{{
    legend:{{position:'top'}},
    tooltip:{{callbacks:{{
      label: ctx => ` ${{ctx.dataset.label}}: $${{ctx.parsed.y.toFixed(2)}}B`
    }}}}
  }},
  scales:{{
    x:{{ type:'linear',
         title:{{display:true, text:'Fiscal Year'}},
         ticks:{{callback: v => v.toFixed(0), stepSize:1}} }},
    y:{{ title:{{display:true, text:'Expenditure (Billions USD)'}},
         ticks:{{callback: v => '$'+v+'B'}} }}
  }}
}};

// Best model charts
new Chart(document.getElementById('natBestChart'), {{
  type:'line',
  data:{{ datasets:[
    mkDS('Historical', natHist, '#1f77b4', []),
    mkDS('Forecast ({nat_best})', natBestFc, '#ff7f0e', [6,3]),
    ...mkCI(natBestCI, '#ff7f0e')
  ]}},
  options: OPT
}});

new Chart(document.getElementById('nyBestChart'), {{
  type:'line',
  data:{{ datasets:[
    mkDS('Historical', nyHist, '#d62728', []),
    mkDS('Forecast ({ny_best})', nyBestFc, '#ff7f0e', [6,3]),
    ...mkCI(nyBestCI, '#ff7f0e')
  ]}},
  options: OPT
}});

// All models charts
const natAllChart = new Chart(document.getElementById('natAllChart'), {{
  type:'line',
  data:{{ datasets:[
    mkDS('Historical',       natHist,   '#1f77b4', []),
    mkDS('Linear',           natLinFc,  '#ff7f0e', [5,3]),
    mkDS('Polynomial (d=2)', natPolyFc, '#2ca02c', [5,3]),
    mkDS("Holt's",           natHwFc,   '#9467bd', [5,3]),
    mkDS('ARIMA(1,1,1)',      natArFc,   '#8c564b', [5,3]),
  ]}},
  options: OPT
}});

const nyAllChart = new Chart(document.getElementById('nyAllChart'), {{
  type:'line',
  data:{{ datasets:[
    mkDS('Historical',       nyHist,   '#d62728', []),
    mkDS('Linear',           nyLinFc,  '#ff7f0e', [5,3]),
    mkDS('Polynomial (d=2)', nyPolyFc, '#2ca02c', [5,3]),
    mkDS("Holt's",           nyHwFc,   '#9467bd', [5,3]),
    mkDS('ARIMA(1,1,1)',      nyArFc,   '#8c564b', [5,3]),
  ]}},
  options: OPT
}});

// Tab switching
function switchTab(id, btn) {{
  document.querySelectorAll('.tab-panel')
    .forEach(p => p.classList.remove('active'));
  document.querySelectorAll('.tab-btn')
    .forEach(b => b.classList.remove('active'));
  document.getElementById('tab-'+id).classList.add('active');
  btn.classList.add('active');
}}

// Model toggle for national
function toggleNat(which, btn) {{
  document.querySelectorAll('#tab-national .subtab-btn')
    .forEach(b => b.classList.remove('active'));
  btn.classList.add('active');
  const keys = ['linear','poly','holt','arima'];
  natAllChart.data.datasets.forEach((ds, i) => {{
    if (i === 0) {{ ds.hidden = false; return; }}
    ds.hidden = !(which === 'all' || which === keys[i-1]);
  }});
  natAllChart.update();
}}

// Model toggle for New York
function toggleNy(which, btn) {{
  document.querySelectorAll('#tab-newyork .subtab-btn')
    .forEach(b => b.classList.remove('active'));
  btn.classList.add('active');
  const keys = ['linear','poly','holt','arima'];
  nyAllChart.data.datasets.forEach((ds, i) => {{
    if (i === 0) {{ ds.hidden = false; return; }}
    ds.hidden = !(which === 'all' || which === keys[i-1]);
  }});
  nyAllChart.update();
}}
</script>
</body>
</html>"""

html_path = f"{OUT_DIR}/Medicaid_Interactive_Dashboard.html"
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html)

print("Saved: Medicaid_Interactive_Dashboard.html")
print("\n✅ Interactive dashboard complete")
print("\nOpen the HTML file in your browser to view it!")

Data prepared for dashboard...
Building HTML...
Saved: Medicaid_Interactive_Dashboard.html

✅ Interactive dashboard complete

Open the HTML file in your browser to view it!


In [24]:
# ============================================================
# Ineractive Dashboard
# ============================================================

def series_to_js(years, values):
    pts = [{"x": int(y), "y": round(float(v)/1e9, 3)} 
           for y, v in zip(years, values)]
    import json
    return json.dumps(pts)

def ci_to_js(years, lower, upper):
    import json
    pts = [{"x": int(y), "lo": round(float(l)/1e9, 3), "hi": round(float(u)/1e9, 3)}
           for y, l, u in zip(years, lower, upper)]
    return json.dumps(pts)

# Rebuild all JS data
nat_hist_js    = series_to_js(national_ts['Year'], national_ts['Expenditure'])
ny_hist_js     = series_to_js(ny_ts['Year'],       ny_ts['Expenditure'])

nat_lin_js     = series_to_js(nat_lin_fc['Year'],  nat_lin_fc['Forecast'])
nat_poly_js    = series_to_js(nat_poly_fc['Year'], nat_poly_fc['Forecast'])
nat_hw_js      = series_to_js(nat_hw_fc['Year'],   nat_hw_fc['Forecast'])
nat_ar_js      = series_to_js(nat_ar_fc['Year'],   nat_ar_fc['Forecast'])
nat_best_js    = series_to_js(nat_best_fc['Year'], nat_best_fc['Forecast'])
nat_best_ci_js = ci_to_js(nat_best_fc['Year'],
                           nat_best_fc['Lower_95'],
                           nat_best_fc['Upper_95'])

ny_lin_js      = series_to_js(ny_lin_fc['Year'],   ny_lin_fc['Forecast'])
ny_poly_js     = series_to_js(ny_poly_fc['Year'],  ny_poly_fc['Forecast'])
ny_hw_js       = series_to_js(ny_hw_fc['Year'],    ny_hw_fc['Forecast'])
ny_ar_js       = series_to_js(ny_ar_fc['Year'],    ny_ar_fc['Forecast'])
ny_best_js     = series_to_js(ny_best_fc['Year'],  ny_best_fc['Forecast'])
ny_best_ci_js  = ci_to_js(ny_best_fc['Year'],
                           ny_best_fc['Lower_95'],
                           ny_best_fc['Upper_95'])

def comp_to_html(comp_df):
    rows = ""
    for _, r in comp_df.iterrows():
        bold = ' style="font-weight:700;background:#fffde7;"' if r['Rank']==1 else ''
        rows += (f'<tr{bold}><td>{int(r["Rank"])}</td>'
                 f'<td>{r["Model"]}</td>'
                 f'<td>${r["MAE (Billions)"]:.2f}B</td>'
                 f'<td>${r["RMSE (Billions)"]:.2f}B</td>'
                 f'<td>{r["MAPE (%)"]:.1f}%</td></tr>')
    return rows

def fc_to_html(fc_df):
    rows = ""
    for _, r in fc_df.iterrows():
        rows += (f'<tr><td>{int(r["Fiscal_Year"])}</td>'
                 f'<td>${r["Forecast_Billions"]:.2f}B</td>'
                 f'<td>${r["Lower_95_CI_Billions"]:.2f}B</td>'
                 f'<td>${r["Upper_95_CI_Billions"]:.2f}B</td></tr>')
    return rows

nat_comp_rows = comp_to_html(nat_comp)
ny_comp_rows  = comp_to_html(ny_comp)
nat_fc_rows   = fc_to_html(nat_export)
ny_fc_rows    = fc_to_html(ny_export)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>Medicaid Expenditure Forecast 2026-2035</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
  *{{box-sizing:border-box;margin:0;padding:0;}}
  body{{font-family:'Segoe UI',Arial,sans-serif;background:#f4f6fa;color:#222;}}
  header{{background:linear-gradient(135deg,#1565c0,#0d47a1);color:#fff;padding:28px 36px;}}
  header h1{{font-size:1.7rem;font-weight:700;}}
  header p{{margin-top:6px;opacity:0.85;font-size:0.95rem;}}
  .container{{max-width:1180px;margin:0 auto;padding:28px 24px;}}
  .tabs{{display:flex;gap:8px;margin-bottom:20px;flex-wrap:wrap;}}
  .tab-btn{{padding:9px 22px;border:none;border-radius:6px;cursor:pointer;
    font-size:0.95rem;font-weight:600;background:#e3eaf6;color:#1565c0;transition:all .2s;}}
  .tab-btn.active{{background:#1565c0;color:#fff;}}
  .tab-btn:hover:not(.active){{background:#c5d8f5;}}
  .tab-panel{{display:none;}}
  .tab-panel.active{{display:block;}}
  .card{{background:#fff;border-radius:10px;box-shadow:0 2px 12px rgba(0,0,0,.08);
    padding:24px;margin-bottom:24px;}}
  .card h2{{font-size:1.1rem;color:#1565c0;margin-bottom:16px;
    border-bottom:2px solid #e3eaf6;padding-bottom:8px;}}
  .chart-wrap{{position:relative;height:380px;}}
  .subtabs{{display:flex;gap:6px;margin-bottom:14px;flex-wrap:wrap;}}
  .subtab-btn{{padding:6px 16px;border:1px solid #c5d8f5;border-radius:20px;
    cursor:pointer;font-size:0.87rem;background:#f0f5ff;color:#1565c0;}}
  .subtab-btn.active{{background:#1565c0;color:#fff;border-color:#1565c0;}}
  table{{width:100%;border-collapse:collapse;font-size:0.9rem;}}
  th{{background:#1565c0;color:#fff;padding:9px 12px;text-align:left;}}
  td{{padding:8px 12px;border-bottom:1px solid #eee;}}
  tr:hover td{{background:#f0f5ff;}}
  .grid-2{{display:grid;grid-template-columns:1fr 1fr;gap:20px;}}
  .badge{{display:inline-block;padding:2px 10px;border-radius:12px;
    font-size:0.8rem;font-weight:700;background:#e8f5e9;color:#2e7d32;}}
  footer{{text-align:center;padding:20px;color:#888;font-size:0.82rem;}}
  @media(max-width:700px){{.grid-2{{grid-template-columns:1fr;}}}}
</style>
</head>
<body>
<header>
  <h1>Medicaid Expenditure Projection Model</h1>
  <p>Historical Analysis (FY 2013-2025) and 10-Year Forecast (2026-2035)
     | National and New York State | Author: Asmita Thapa</p>
</header>

<div class="container">
  <div class="tabs">
    <button class="tab-btn active" onclick="switchTab('national',this)">National</button>
    <button class="tab-btn"        onclick="switchTab('newyork',this)">New York State</button>
    <button class="tab-btn"        onclick="switchTab('compare',this)">Model Comparison</button>
    <button class="tab-btn"        onclick="switchTab('tables',this)">Forecast Tables</button>
  </div>

  <div id="tab-national" class="tab-panel active">
    <div class="card">
      <h2>National Medicaid Expenditure - Best Model Forecast
        <span class="badge">{nat_best}</span></h2>
      <div class="chart-wrap"><canvas id="natBestChart"></canvas></div>
    </div>
    <div class="card">
      <h2>National - All Models Comparison</h2>
      <div class="subtabs">
        <button class="subtab-btn active" onclick="toggleNat('all',this)">All Models</button>
        <button class="subtab-btn" onclick="toggleNat('linear',this)">Linear</button>
        <button class="subtab-btn" onclick="toggleNat('poly',this)">Polynomial</button>
        <button class="subtab-btn" onclick="toggleNat('holt',this)">Holt's</button>
        <button class="subtab-btn" onclick="toggleNat('arima',this)">ARIMA</button>
      </div>
      <div class="chart-wrap"><canvas id="natAllChart"></canvas></div>
    </div>
  </div>

  <div id="tab-newyork" class="tab-panel">
    <div class="card">
      <h2>New York State Medicaid Expenditure - Best Model Forecast
        <span class="badge">{ny_best}</span></h2>
      <div class="chart-wrap"><canvas id="nyBestChart"></canvas></div>
    </div>
    <div class="card">
      <h2>New York - All Models Comparison</h2>
      <div class="subtabs">
        <button class="subtab-btn active" onclick="toggleNy('all',this)">All Models</button>
        <button class="subtab-btn" onclick="toggleNy('linear',this)">Linear</button>
        <button class="subtab-btn" onclick="toggleNy('poly',this)">Polynomial</button>
        <button class="subtab-btn" onclick="toggleNy('holt',this)">Holt's</button>
        <button class="subtab-btn" onclick="toggleNy('arima',this)">ARIMA</button>
      </div>
      <div class="chart-wrap"><canvas id="nyAllChart"></canvas></div>
    </div>
  </div>

  <div id="tab-compare" class="tab-panel">
    <div class="grid-2">
      <div class="card">
        <h2>National - Model Performance (Test: FY 2022-2025)</h2>
        <table>
          <thead><tr><th>Rank</th><th>Model</th>
            <th>MAE</th><th>RMSE</th><th>MAPE</th></tr></thead>
          <tbody>{nat_comp_rows}</tbody>
        </table>
      </div>
      <div class="card">
        <h2>New York - Model Performance (Test: FY 2022-2025)</h2>
        <table>
          <thead><tr><th>Rank</th><th>Model</th>
            <th>MAE</th><th>RMSE</th><th>MAPE</th></tr></thead>
          <tbody>{ny_comp_rows}</tbody>
        </table>
      </div>
    </div>
    <div class="card">
      <h2>Methodology Notes</h2>
      <p style="line-height:1.8;font-size:0.93rem;">
        <strong>Train/Test Split:</strong> Models trained on FY 2013-2021,
        evaluated on FY 2022-2025.<br>
        <strong>Linear Regression:</strong> OLS on Year. Prediction intervals
        widen using t-distribution leverage formula.<br>
        <strong>Polynomial Regression (d=2):</strong> Quadratic OLS.<br>
        <strong>Holt's Exponential Smoothing:</strong> Level and trend parameters
        optimised by minimising SSE. CI widens as sqrt(h).<br>
        <strong>ARIMA(1,1,1):</strong> AR and MA parameters estimated by
        conditional least squares on first differences.<br>
        <strong>Best model</strong> selected by lowest MAPE on hold-out test set.
      </p>
    </div>
  </div>

  <div id="tab-tables" class="tab-panel">
    <div class="grid-2">
      <div class="card">
        <h2>National Forecast 2026-2035
          <span class="badge">{nat_best}</span></h2>
        <table>
          <thead><tr><th>Year</th><th>Forecast</th>
            <th>Lower 95% CI</th><th>Upper 95% CI</th></tr></thead>
          <tbody>{nat_fc_rows}</tbody>
        </table>
      </div>
      <div class="card">
        <h2>New York Forecast 2026-2035
          <span class="badge">{ny_best}</span></h2>
        <table>
          <thead><tr><th>Year</th><th>Forecast</th>
            <th>Lower 95% CI</th><th>Upper 95% CI</th></tr></thead>
          <tbody>{ny_fc_rows}</tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<footer>Medicaid Expenditure Projection Model | FY 2013-2025 |
  Models: Linear, Polynomial, Holt's, ARIMA(1,1,1) | Author: Asmita Thapa</footer>

<script>
const natHist    = {nat_hist_js};
const nyHist     = {ny_hist_js};
const natLinFc   = {nat_lin_js};
const natPolyFc  = {nat_poly_js};
const natHwFc    = {nat_hw_js};
const natArFc    = {nat_ar_js};
const natBestFc  = {nat_best_js};
const natBestCI  = {nat_best_ci_js};
const nyLinFc    = {ny_lin_js};
const nyPolyFc   = {ny_poly_js};
const nyHwFc     = {ny_hw_js};
const nyArFc     = {ny_ar_js};
const nyBestFc   = {ny_best_js};
const nyBestCI   = {ny_best_ci_js};

function mkDS(label, pts, color, dash) {{
  return {{
    label,
    data: pts,
    borderColor: color,
    backgroundColor: color,
    borderWidth: 2.2,
    borderDash: dash || [],
    pointRadius: 5,
    tension: 0.25
  }};
}}

function mkCI(pts, color) {{
  return [
    {{ label:'Upper CI',
       data: pts.map(p => ({{x:p.x, y:p.hi}})),
       borderColor:'transparent',
       backgroundColor: color+'28',
       borderWidth:0, pointRadius:0,
       fill:'+1', tension:0.25 }},
    {{ label:'Lower CI',
       data: pts.map(p => ({{x:p.x, y:p.lo}})),
       borderColor:'transparent',
       backgroundColor: color+'28',
       borderWidth:0, pointRadius:0,
       fill:false, tension:0.25 }}
  ];
}}

const OPT = {{
  responsive:true,
  maintainAspectRatio:false,
  interaction:{{mode:'index', intersect:false}},
  plugins:{{
    legend:{{position:'top'}},
    tooltip:{{callbacks:{{
      label: ctx => ` ${{ctx.dataset.label}}: $${{ctx.parsed.y.toFixed(2)}}B`
    }}}}
  }},
  scales:{{
    x:{{ type:'linear',
         title:{{display:true, text:'Fiscal Year'}},
         ticks:{{callback: v => v.toFixed(0), stepSize:2}} }},
    y:{{ title:{{display:true, text:'Expenditure (Billions USD)'}},
         ticks:{{callback: v => '$'+v+'B'}} }}
  }}
}};

new Chart(document.getElementById('natBestChart'), {{
  type:'line',
  data:{{ datasets:[
    mkDS('Historical', natHist, '#1f77b4', []),
    mkDS('Forecast ({nat_best})', natBestFc, '#ff7f0e', [6,3]),
    ...mkCI(natBestCI, '#ff7f0e')
  ]}},
  options: OPT
}});

new Chart(document.getElementById('nyBestChart'), {{
  type:'line',
  data:{{ datasets:[
    mkDS('Historical', nyHist, '#d62728', []),
    mkDS('Forecast ({ny_best})', nyBestFc, '#ff7f0e', [6,3]),
    ...mkCI(nyBestCI, '#ff7f0e')
  ]}},
  options: OPT
}});

const natAllChart = new Chart(document.getElementById('natAllChart'), {{
  type:'line',
  data:{{ datasets:[
    mkDS('Historical',       natHist,   '#1f77b4', []),
    mkDS('Linear',           natLinFc,  '#ff7f0e', [5,3]),
    mkDS('Polynomial (d=2)', natPolyFc, '#2ca02c', [5,3]),
    mkDS("Holt's",           natHwFc,   '#9467bd', [5,3]),
    mkDS('ARIMA(1,1,1)',      natArFc,   '#8c564b', [5,3]),
  ]}},
  options: OPT
}});

const nyAllChart = new Chart(document.getElementById('nyAllChart'), {{
  type:'line',
  data:{{ datasets:[
    mkDS('Historical',       nyHist,   '#d62728', []),
    mkDS('Linear',           nyLinFc,  '#ff7f0e', [5,3]),
    mkDS('Polynomial (d=2)', nyPolyFc, '#2ca02c', [5,3]),
    mkDS("Holt's",           nyHwFc,   '#9467bd', [5,3]),
    mkDS('ARIMA(1,1,1)',      nyArFc,   '#8c564b', [5,3]),
  ]}},
  options: OPT
}});

function switchTab(id, btn) {{
  document.querySelectorAll('.tab-panel')
    .forEach(p => p.classList.remove('active'));
  document.querySelectorAll('.tab-btn')
    .forEach(b => b.classList.remove('active'));
  document.getElementById('tab-'+id).classList.add('active');
  btn.classList.add('active');
}}

function toggleNat(which, btn) {{
  document.querySelectorAll('#tab-national .subtab-btn')
    .forEach(b => b.classList.remove('active'));
  btn.classList.add('active');
  const keys = ['linear','poly','holt','arima'];
  natAllChart.data.datasets.forEach((ds, i) => {{
    if (i === 0) {{ ds.hidden = false; return; }}
    ds.hidden = !(which === 'all' || which === keys[i-1]);
  }});
  natAllChart.update();
}}

function toggleNy(which, btn) {{
  document.querySelectorAll('#tab-newyork .subtab-btn')
    .forEach(b => b.classList.remove('active'));
  btn.classList.add('active');
  const keys = ['linear','poly','holt','arima'];
  nyAllChart.data.datasets.forEach((ds, i) => {{
    if (i === 0) {{ ds.hidden = false; return; }}
    ds.hidden = !(which === 'all' || which === keys[i-1]);
  }});
  nyAllChart.update();
}}
</script>
</body>
</html>"""

html_path = f"{OUT_DIR}/Medicaid_Interactive_Dashboard.html"
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html)

print("Saved: Medicaid_Interactive_Dashboard.html")
print("Open the file in your browser to view it!")
print("\n✅ Dashboard complete")

Saved: Medicaid_Interactive_Dashboard.html
Open the file in your browser to view it!

✅ Dashboard complete


In [25]:
# ============================================================
# GENERATE README.md
# ============================================================

lines = []
lines.append("# Medicaid Expenditure Projection Model")
lines.append("### National + New York State | FY 2013-2025 -> Forecast 2026-2035")
lines.append("")
lines.append("**Author:** Asmita Thapa  ")
lines.append("**Date:** April 2026  ")
lines.append("")
lines.append("---")
lines.append("")
lines.append("## Overview")
lines.append("")
lines.append("This project builds a data-driven forecasting tool that analyzes 13 years of historical Medicaid expenditure data (FY 2013-2025) and generates 10-year projections (2026-2035) with 95% confidence intervals for both the United States nationally and New York State. Four forecasting models are built, evaluated, and compared. The best-performing model for each series is selected based on a hold-out test set evaluation.")
lines.append("")
lines.append("The outputs are designed to support policy planning, budget allocation, and healthcare resource forecasting decisions.")
lines.append("")
lines.append("---")
lines.append("")
lines.append("## Project Structure")
lines.append("")
lines.append("    medicaid_project/")
lines.append("    |-- medicaid_forecast.ipynb")
lines.append("    |-- FY_2013_MFCU_Statistical_Chart.xlsx")
lines.append("    |-- FY_2014_MFCU_Statistical_Chart.xlsx")
lines.append("    |-- ... (FY 2015-2025)")
lines.append("    |-- outputs/")
lines.append("        |-- 01_EDA_Overview.png")
lines.append("        |-- 02_Decomposition.png")
lines.append("        |-- 03_Forecast_National.png")
lines.append("        |-- 04_Forecast_NewYork.png")
lines.append("        |-- 05_AllModels_National.png")
lines.append("        |-- 06_AllModels_NewYork.png")
lines.append("        |-- Medicaid_Expenditure_Forecast_2026_2035.xlsx")
lines.append("        |-- National_Forecast_2026_2035.csv")
lines.append("        |-- NewYork_Forecast_2026_2035.csv")
lines.append("        |-- Medicaid_Interactive_Dashboard.html")
lines.append("        |-- README.md")
lines.append("")
lines.append("---")
lines.append("")
lines.append("## Setup and Installation")
lines.append("")
lines.append("Install required libraries:")
lines.append("")
lines.append("    pip install numpy pandas matplotlib scikit-learn scipy openpyxl jupyter")
lines.append("")
lines.append("No additional packages required. ARIMA and Holt's Exponential Smoothing are implemented from first principles using numpy and scipy.")
lines.append("")
lines.append("Run the notebook:")
lines.append("")
lines.append("    jupyter notebook medicaid_forecast.ipynb")
lines.append("")
lines.append("Before running, update the DATA_DIR variable in the second cell:")
lines.append("")
lines.append("    DATA_DIR = 'your/path/to/data/folder'")
lines.append("")
lines.append("---")
lines.append("")
lines.append("## Data Source")
lines.append("")
lines.append("The dataset consists of 13 annual Excel files published by the U.S. Department of Health and Human Services, Office of Inspector General (OIG). Each file contains state-level Medicaid expenditure data for one fiscal year.")
lines.append("")
lines.append("Files used: FY_2013_MFCU_Statistical_Chart.xlsx through FY_2025_MFCU_Statistical_Chart.xlsx")
lines.append("")
lines.append("Key field extracted: Total Medicaid Expenditures (state and federal shares combined)")
lines.append("")
lines.append("---")
lines.append("")
lines.append("## Data Cleaning Challenges")
lines.append("")
lines.append("The raw Excel files presented several structural inconsistencies:")
lines.append("")
lines.append("| Challenge | Solution |")
lines.append("|---|---|")
lines.append("| FY 2019 uses State1 as column name | Header row detected dynamically by keyword scan |")
lines.append("| FY 2016 contains a Grand Total summary row | Filtered using skip-prefix logic |")
lines.append("| FY 2024 Total cell contains =SUM() formula | Total row excluded; state rows summed directly |")
lines.append("| States count changes from 50 to 53 after FY 2019 | DC, Puerto Rico, U.S. Virgin Islands added — noted in output |")
lines.append("| Original code double-counted rows | Fixed by extracting only state-level rows |")
lines.append("")
lines.append("---")
lines.append("")
lines.append("## Methodology")
lines.append("")
lines.append("### 1. Data Ingestion and Cleaning")
lines.append("- All 13 Excel files parsed using openpyxl in read-only mode")
lines.append("- Header row located dynamically — handles all naming variants across years")
lines.append("- Total Medicaid Expenditures column identified by keyword match")
lines.append("- Total and footnote rows excluded by prefix filtering")
lines.append("- All values converted to float; blanks and invalid entries dropped")
lines.append("")
lines.append("### 2. Time Series Construction")
lines.append("- National: sum of all state and jurisdiction expenditures per fiscal year")
lines.append("- New York: filtered directly from state-level records")
lines.append("- Year-over-year growth rates computed for EDA")
lines.append("")
lines.append("### 3. Exploratory Data Analysis")
lines.append("- Historical trend plots for National and New York")
lines.append("- Year-over-year growth rate comparison")
lines.append("- New York share of national expenditure over time")
lines.append("- Time series decomposition into trend and residual components")
lines.append("- Descriptive statistics including min, max, mean, median, std deviation")
lines.append("")
lines.append("### 4. Forecasting Models")
lines.append("")
lines.append("All models use a train/test split:")
lines.append("- Training set: FY 2013-2021")
lines.append("- Test set: FY 2022-2025 (hold-out, never seen during training)")
lines.append("")
lines.append("| Model | Description |")
lines.append("|---|---|")
lines.append("| Linear Regression | OLS on Year as predictor. Prediction intervals use t-distribution leverage formula that widens with forecast horizon. |")
lines.append("| Polynomial Regression (d=2) | Quadratic OLS. Captures non-linear acceleration in spending growth. |")
lines.append("| Holt's Exponential Smoothing | Double exponential smoothing with level (alpha) and trend (beta) optimised by minimising SSE. CI widens as sqrt(h). |")
lines.append("| ARIMA(1,1,1) | AR and MA parameters estimated by conditional least squares on first differences. Uncertainty grows as sqrt(h). |")
lines.append("")
lines.append("### 5. Model Evaluation")
lines.append("")
lines.append("| Metric | Description |")
lines.append("|---|---|")
lines.append("| MAE | Mean Absolute Error |")
lines.append("| RMSE | Root Mean Squared Error |")
lines.append("| MAPE | Mean Absolute Percentage Error — used for model selection |")
lines.append("")
lines.append("---")
lines.append("")
lines.append("## Results")
lines.append("")
lines.append("### Model Performance — National (Test: FY 2022-2025)")
lines.append("")
lines.append("| Rank | Model | MAE | RMSE | MAPE |")
lines.append("|---|---|---|---|---|")
lines.append("| 1 | ARIMA(1,1,1) | $33.7B | $35.5B | 3.6% |")
lines.append("| 2 | Holt's Exponential Smoothing | $107.6B | $113.8B | 11.4% |")
lines.append("| 3 | Polynomial Regression (d=2) | $120.4B | $126.1B | 12.8% |")
lines.append("| 4 | Linear Regression | $129.9B | $136.2B | 13.8% |")
lines.append("")
lines.append("### Model Performance — New York (Test: FY 2022-2025)")
lines.append("")
lines.append("| Rank | Model | MAE | RMSE | MAPE |")
lines.append("|---|---|---|---|---|")
lines.append("| 1 | Linear Regression | $13.0B | $13.9B | 13.4% |")
lines.append("| 2 | Holt's Exponential Smoothing | $20.7B | $21.8B | 21.4% |")
lines.append("| 3 | ARIMA(1,1,1) | $22.0B | $23.2B | 22.8% |")
lines.append("| 4 | Polynomial Regression (d=2) | $29.9B | $31.9B | 30.8% |")
lines.append("")
lines.append("### National 10-Year Forecast — ARIMA(1,1,1)")
lines.append("")
lines.append("| Fiscal Year | Forecast | Lower 95% CI | Upper 95% CI |")
lines.append("|---|---|---|---|")
lines.append("| 2026 | $1,112.6B | $1,055.9B | $1,169.2B |")
lines.append("| 2027 | $1,192.6B | $1,112.4B | $1,272.8B |")
lines.append("| 2028 | $1,271.8B | $1,173.6B | $1,370.0B |")
lines.append("| 2029 | $1,350.3B | $1,236.9B | $1,463.6B |")
lines.append("| 2030 | $1,427.9B | $1,301.2B | $1,554.7B |")
lines.append("| 2031 | $1,504.8B | $1,366.0B | $1,643.7B |")
lines.append("| 2032 | $1,580.9B | $1,431.0B | $1,730.9B |")
lines.append("| 2033 | $1,656.3B | $1,496.0B | $1,816.6B |")
lines.append("| 2034 | $1,730.9B | $1,560.8B | $1,900.9B |")
lines.append("| 2035 | $1,804.7B | $1,625.5B | $1,984.0B |")
lines.append("")
lines.append("### New York 10-Year Forecast — Linear Regression")
lines.append("")
lines.append("| Fiscal Year | Forecast | Lower 95% CI | Upper 95% CI |")
lines.append("|---|---|---|---|")
lines.append("| 2026 | $87.6B | $61.3B | $114.0B |")
lines.append("| 2027 | $90.0B | $62.1B | $118.0B |")
lines.append("| 2028 | $92.5B | $62.8B | $122.2B |")
lines.append("| 2029 | $94.9B | $63.5B | $126.4B |")
lines.append("| 2030 | $97.4B | $64.1B | $130.7B |")
lines.append("| 2031 | $99.8B | $64.7B | $134.9B |")
lines.append("| 2032 | $102.2B | $65.2B | $139.3B |")
lines.append("| 2033 | $104.7B | $65.8B | $143.6B |")
lines.append("| 2034 | $107.1B | $66.3B | $148.0B |")
lines.append("| 2035 | $109.6B | $66.7B | $152.4B |")
lines.append("")
lines.append("---")
lines.append("")
lines.append("## Key Observations")
lines.append("")
lines.append("- National Medicaid spending nearly doubled from $453B in FY 2013 to $1.03T in FY 2025, an average annual growth rate of 7.2%.")
lines.append("- New York grew from $54B to $103B over the same period at 6.1% average annual growth.")
lines.append("- ARIMA(1,1,1) outperforms all models nationally with MAPE of 3.6%, because national Medicaid spending follows strong year-to-year momentum.")
lines.append("- Linear Regression performs best for New York (MAPE 13.4%) because NY follows a consistent long-term upward trend despite some volatility.")
lines.append("- Polynomial Regression failed for New York, producing negative forecasts by 2033 due to overfitting. This highlights why hold-out test evaluation is essential.")
lines.append("- FY 2019 shows a notable dip in New York spending ($60.2B vs $75.3B in FY 2018), likely reflecting a policy or reporting change. Retained in analysis as real observed data.")
lines.append("- The COVID-19 period (FY 2020-2021) shows acceleration in national spending driven by Medicaid enrollment expansion.")
lines.append("")
lines.append("---")
lines.append("")
lines.append("## Deliverables")
lines.append("")
lines.append("| File | Description |")
lines.append("|---|---|")
lines.append("| medicaid_forecast.ipynb | Full reproducible Jupyter notebook |")
lines.append("| 01_EDA_Overview.png | 4-panel exploratory data analysis |")
lines.append("| 02_Decomposition.png | Time series decomposition |")
lines.append("| 03_Forecast_National.png | National best-model forecast with 95% CI |")
lines.append("| 04_Forecast_NewYork.png | New York best-model forecast with 95% CI |")
lines.append("| 05_AllModels_National.png | All 4 models overlaid — national |")
lines.append("| 06_AllModels_NewYork.png | All 4 models overlaid — New York |")
lines.append("| Medicaid_Expenditure_Forecast_2026_2035.xlsx | 7-sheet Excel with forecasts, model comparison, and historical data |")
lines.append("| National_Forecast_2026_2035.csv | National forecast table |")
lines.append("| NewYork_Forecast_2026_2035.csv | New York forecast table |")
lines.append("| Medicaid_Interactive_Dashboard.html | Interactive dashboard — open in Chrome or Edge |")

readme_path = f"{OUT_DIR}/README.md"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f"Saved: README.md")
print(f"Location: {readme_path}")
print("\n✅ README complete")

Saved: README.md
Location: C:/Users/User/Documents/EMRTS_ProjectV2\outputs/README.md

✅ README complete
